# 4. Feature Engineering

Feature engineering is the art of creating informative features from raw data. Good features can dramatically improve model performance. This notebook covers:
- Polynomial and interaction features
- Ratio and mathematical features
- Text-derived features
- Aggregation features
- Feature selection with mutual information

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

## 4.1 Load the Titanic Dataset

In [ ]:
url = ("https://raw.githubusercontent.com/datasciencedojo/"
       "datasets/master/titanic.csv")
df = pd.read_csv(url)
print(f"Original shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

## 4.2 Polynomial Features

Polynomial features capture **non-linear relationships** and **interactions** between variables. A degree-2 expansion of $[x_1, x_2]$ produces $[x_1, x_2, x_1^2, x_1 x_2, x_2^2]$.

In [ ]:
features = df[['Age', 'Fare']].dropna()
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(features)
poly_names = poly.get_feature_names_out(['Age', 'Fare'])

print(f"Original: {features.shape[1]} features")
print(f"Polynomial (degree 2): {len(poly_names)} features")
print(f"Names: {poly_names}")

## 4.3 Ratio and Mathematical Features

Domain knowledge often suggests meaningful ratios and combinations. For Titanic:
- **FamilySize** = SibSp + Parch + 1
- **FarePerPerson** = Fare / FamilySize
- **Log transform** of skewed features

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['FarePerPerson'] = df['Fare'] / df['FamilySize']
df['Fare_log'] = np.log1p(df['Fare'])

print("Survival rate by FamilySize:")
print(df.groupby('FamilySize')['Survived'].mean().round(3))

## 4.4 Text-Derived Features

Even simple text processing can yield powerful features. We extract the **title** from passenger names using regex.

In [ ]:
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)

title_map = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
}
df['Title_clean'] = df['Title'].map(title_map).fillna('Rare')

print("Survival by title:")
print(df.groupby('Title_clean')['Survived'].mean()
        .sort_values(ascending=False))

## 4.5 Cabin and Aggregation Features

- **HasCabin**: a binary indicator can capture the pattern that cabin-assigned passengers had higher survival
- **Aggregation features**: compare individual values to group statistics

In [ ]:
df['HasCabin'] = df['Cabin'].notnull().astype(int)

# Average fare by class
class_stats = df.groupby('Pclass')['Fare'].agg(
    class_mean_fare='mean').reset_index()
df = df.merge(class_stats, on='Pclass', how='left')
df['FareVsClass'] = df['Fare'] / df['class_mean_fare']

print(df[['Pclass', 'Fare', 'class_mean_fare', 'FareVsClass']].head(10))

## 4.6 Interaction Features

Combining categorical and numerical features can reveal patterns that neither variable captures alone.

In [ ]:
df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)
print("Survival by Sex x Pclass:")
print(df.groupby('Sex_Pclass')['Survived'].mean()
        .sort_values(ascending=False))

## 4.7 Feature Selection with Mutual Information

**Mutual information** measures how much knowing a feature reduces uncertainty about the target. It captures both linear and non-linear dependencies.

In [ ]:
feature_cols = ['Age', 'Fare', 'FamilySize', 'IsAlone',
                'FarePerPerson', 'HasCabin', 'Fare_log', 'FareVsClass']

subset = df[feature_cols + ['Survived']].dropna()
X = subset[feature_cols]
y = subset['Survived']

mi = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi, index=feature_cols).sort_values(ascending=False)

print("Mutual information with Survived:")
for feat, score in mi_series.items():
    bar = '#' * int(score * 50)
    print(f"  {feat:<20} {score:.4f} {bar}")

## Key Takeaways

1. **Domain knowledge** is the most important ingredient in feature engineering
2. **Polynomial features** capture non-linear effects but increase dimensionality
3. **Text features** (titles, patterns) can be highly predictive
4. **Aggregation features** compare individuals to their group
5. **Mutual information** helps rank features by predictive power